# Single-Spin Metropolis algorithm (checkerboard) on 2D Ising Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit
from tqdm import tqdm
import os

In [ ]:
@njit
def initial_state(L, random_start=True):
    state = np.empty((L, L), dtype=np.int8)
    if random_start:
        # Random ±1 using threshold
        for i in range(L):
            for j in range(L):
                if np.random.random() < 0.5:
                    state[i, j] = 1
                else:
                    state[i, j] = -1
    else:
        # All spins up (+1)
        state[:] = 1
    return state

In [ ]:
@njit(cache=True)
def calc_total_energy(state, J, h):
    # Only used once at the start to initialize the counter
    H = 0.0
    rows, cols = state.shape
    for i in range(rows):
        for j in range(cols):
            s = state[i, j]
            neighbor_sum = state[i, (j + 1) % cols] + state[(i + 1) % rows, j]
            H -= J * s * neighbor_sum
            H -= h * s
    return H

In [ ]:
@njit(cache=True)
def get_magnetization(state):
    return np.sum(state)

In [ ]:
def compute_susceptibility(Ms, beta, N, h=0):
    m = Ms / N
    m2_mean = np.mean(m**2)
    
    if h == 0:
        m_mean = np.mean(np.abs(m))
    else:
        m_mean = np.mean(m)
        
    return beta * N * (m2_mean - m_mean**2)

In [ ]:
@njit(cache=True)
def run_simulation_incremental(state, J, h, beta, n_sweeps, record_interval, initial_E):
    rows, cols = state.shape
    current_E = initial_E
    
    n_records = n_sweeps // record_interval
    energies = np.zeros(n_records, dtype=np.float64)
    magnetizations = np.zeros(n_records, dtype=np.float64)
    rec_idx = 0
    
    for sweep in range(n_sweeps):
        # Checkerboard update
        for parity in range(2):
            for i in range(rows):
                # Iterate strictly over the checkerboard pattern
                for j in range((i + parity) % 2, cols, 2):
                    s = state[i, j]
                    neighbor_sum = (
                        state[(i + 1) % rows, j]
                        + state[(i - 1) % rows, j]
                        + state[i, (j + 1) % cols]
                        + state[i, (j - 1) % cols]
                    )
                    
                    # Calculate dE exactly
                    dE = 2.0 * s * (J * neighbor_sum + h)
                    
                    # Metropolis acceptance
                    if dE <= 0:
                        accept = True
                    else:
                        accept = np.random.random() < np.exp(-beta * dE)
                        
                    if accept:
                        state[i, j] = -s
                        current_E += dE

        if (sweep + 1) % record_interval == 0:
            energies[rec_idx] = current_E
            magnetizations[rec_idx] = get_magnetization(state)
            rec_idx += 1
            
    return energies, magnetizations, state

In [ ]:
def plot_state(state):
    plt.imshow(state, cmap='gray', vmin=-1, vmax=1)
    plt.axis('off')
    plt.show()

def plot_energies_and_magnetizations(energies, magnetizations):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(energies, label='Energy') 
    plt.xlabel('Step')
    plt.ylabel('Energy')
    plt.title('Energy vs. Steps')

    plt.subplot(1, 2, 2)
    plt.plot(magnetizations, label='Magnetization', color='orange')
    plt.xlabel('Step')
    plt.ylabel('Magnetization')
    plt.title('Magnetization vs. Steps')

    plt.tight_layout()
    plt.show()

## Simulation

In [ ]:
base_dir = f"data/Metropolis_Temp_Scan_6_Million_Sweeps"
os.makedirs(base_dir, exist_ok=True)

In [ ]:
L_ls = [128]
h_ls = [0.0]
beta_ls = [0.4407]
J = 1.0

n_sweeps = 6000000

record_interval = 1

In [ ]:
with open(f"{base_dir}/beta_list.txt", "w") as f:
    for beta in beta_ls:
        f.write(f"{beta}\n")

In [ ]:
print(f"Estimating runtime...")
# Run a tiny warm-up to compile JIT
init_st = initial_state(L=16)
init_E = calc_total_energy(init_st, J, 0.0)
run_simulation_incremental(init_st, J, 0.0, 0.4, 100, 1, init_E)
print("JIT compiled.")

Main simulation loop

In [ ]:
for L in tqdm(L_ls, desc="Lattice sizes"):
    L_directory = f"{base_dir}/L_{L}"
    os.makedirs(L_directory, exist_ok=True)
    print(f"\n Starting simulations for L = {L}")

    for h in tqdm(h_ls, desc=f"h values for L={L}"):
        h_dir = f"{L_directory}/h_{h}"
        os.makedirs(h_dir, exist_ok=True)
        print(f"→ External field h = {h}")

        # Write parameter info file once per (L, h)
        param_path = f"{h_dir}/parameters.txt"
        with open(param_path, "w") as f:
            f.write(f"Lattice size: {L}x{L}\n")
            f.write(f"Interaction strength J: {J}\n")
            f.write(f"External field h: {h}\n")
            f.write(f"Number of sweeps (steps = sweeps * L * L): {n_sweeps}\n")
            f.write(f"Record interval: {record_interval}\n")
            f.write(f"Output directory: {h_dir}\n")
            f.write("Data saved as: mag_eng_beta_*.npz\n")
            f.write("Contents: energies, magnetizations\n")
        print(f"Parameters written to {param_path}")

        for beta in tqdm(beta_ls, desc=f"β values for L={L}, h={h}"):
            print(f"Running β = {beta:.4f} ...")

            # Initialize state for this beta
            st = initial_state(L)
            E_start = calc_total_energy(st, J, h)

            # Run simulation
            Es, Ms, final_st = run_simulation_incremental(
                st, J, h, beta, n_sweeps, record_interval, E_start
            )

            # Save results
            out_path = f"{h_dir}/mag_eng_beta_{beta:.4f}.npz"
            np.savez_compressed(out_path, energies=Es, magnetizations=Ms)
            print(f"Saved results → {out_path} [{len(Es)} records]")

        print(f"Finished all β values for h = {h}")
    print(f"Completed all external fields for L = {L}")
print("\n All simulations finished successfully!")